# Baseline SLM Evaluation: Qwen2.5-1.5B-Instruct

**Project:** Python Learning Assistant Using Small Language Model (SLM)  
**Notebook:** `notebooks/baseline_evaluation.ipynb`  
**Target Model:** `Qwen/Qwen2.5-1.5B-Instruct` (Unadapted Pretrained Baseline)  
**Pinned Revision:** `989aa7980e4cf806f80c7fef2b1adb7bc71aa306` (Apache-2.0)  
**Evaluation Dataset:** `data/processed/test.jsonl` (38 Held-Out Test Questions)  
**Target Hardware:** Google Colab FREE Tier GPU (NVIDIA Tesla T4 or any available CUDA GPU). Zero paid APIs or subscriptions required.  
**Output Artifact:** `experiments/baseline_results.json` (prompted for one-click browser download upon completion)  

---
### Step-by-Step Instructions for Google Colab:

1. **Select GPU Runtime:**
   - Navigate to: **Runtime -> Change runtime type**.
   - Set **Hardware accelerator** to **T4 GPU** (Standard Free tier).

2. **Workspace & Dataset Setup (Recommended):**
   - If not already done, clone the repository and switch directory:
     ```bash
     !git clone https://github.com/sasikalascgm-droid/python-learning-assistant-slm.git
     %cd python-learning-assistant-slm
     ```
   - Note: Step 5 includes automatic workspace detection and GitHub fallback recovery so the 38 test cases load reliably regardless of execution folder.

3. **Execution:**
   - Click **Runtime -> Run all**.
   - Step 5 evaluates all 38 test questions.
   - Step 6 compiles latency, throughput, token counts, and exports `experiments/baseline_results.json`.


In [ ]:
# Step 1: Runtime Hardware Audit & Colab Workspace Configuration
import os
import sys
import psutil
import torch
from pathlib import Path

# Auto-detect Google Colab and set working directory if cloned
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
if IN_COLAB:
    for repo_candidate in ['/content/python-learning-assistant-slm', '/content/slm-python-learning-assistant']:
        if (Path(repo_candidate) / 'data/processed/test.jsonl').exists():
            os.chdir(repo_candidate)
            print(f'Switched working directory to: {repo_candidate}')
            break

print('=== Runtime Hardware Audit ===')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'CUDA GPU Detected : {gpu_name}')
    print(f'Available VRAM    : {vram_gb:.2f} GB')
    device = 'cuda'
    major, _ = torch.cuda.get_device_capability(0)
    dtype = torch.bfloat16 if (major >= 8 and torch.cuda.is_bf16_supported()) else torch.float16
else:
    cpu_count = os.cpu_count()
    ram_gb = psutil.virtual_memory().total / (1024**3)
    avail_gb = psutil.virtual_memory().available / (1024**3)
    print(f'No CUDA GPU detected. Running on CPU ({cpu_count} logical cores).')
    print(f'System RAM: {ram_gb:.2f} GB Total, {avail_gb:.2f} GB Available.')
    device = 'cpu'
    dtype = torch.float32

print(f'Active Device: {device}, Precision: {dtype}')


In [ ]:
# Step 2: Install Verified Open-Source Dependencies (Pinning huggingface-hub<2.0 for zero conflict)
!pip install -q "transformers>=5.22.0" "huggingface-hub>=1.31.0,<2.0" "accelerate>=1.15.0" psutil pyyaml


In [ ]:
# Step 2b: Package Compatibility Verification & Conflict Check
import sys
import subprocess

print('=== Installed Package Versions ===')
for pkg in ['torch', 'transformers', 'huggingface_hub', 'accelerate']:
    try:
        mod = __import__(pkg)
        print(f'{pkg:16}: {getattr(mod, "__version__", "unknown")}')
    except ImportError:
        print(f'{pkg:16}: not installed')

print('\n=== Dependency Conflict Audit (pip check) ===')
check_res = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
if check_res.returncode == 0:
    print('PASSED: Zero dependency conflicts detected across installed packages.')
else:
    print(f'Notice from pip check:\n{check_res.stdout.strip()}')


In [ ]:
# Step 3: Load Base Model and Tokenizer (Singleton Initialization with Pinned Revision)
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
PINNED_REVISION = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
print(f'Loading tokenizer and model for "{MODEL_ID}" (revision {PINNED_REVISION})...')

load_start = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=PINNED_REVISION, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=PINNED_REVISION,
    torch_dtype=dtype,
    low_cpu_mem_usage=True
)
if device == 'cuda':
    model = model.to('cuda')
model.eval()

load_time = time.perf_counter() - load_start
print(f'Model successfully loaded in {load_time:.2f} seconds.')


In [ ]:
# Step 4: Define System Prompt and Inference Pipeline
SYSTEM_PROMPT = '''You are an expert, patient, and friendly Python Learning Assistant.
Your mission is to help beginner students master Python programming.
Follow these principles:
1. Explain concepts in clear, simple English without unnecessary jargon.
2. Provide idiomatic, clean, and bug-free Python code examples.
3. Break down code step-by-step or line-by-line when explaining snippets.
4. When diagnosing errors, clearly explain the cause and provide the exact fix.
5. Always encourage the student with beginner-friendly practice questions and hints.
'''

def generate_slm_response(instruction, input_text='', max_new_tokens=512, temperature=0.7, top_p=0.9, repetition_penalty=1.1):
    user_content = instruction.strip()
    if input_text and input_text.strip():
        user_content += f'\n\n```python\n{input_text.strip()}\n```'

    prompt = (
        f'<|im_start|>system\n{SYSTEM_PROMPT.strip()}<|im_end|>\n'
        f'<|im_start|>user\n{user_content}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )

    inputs = tokenizer(prompt, return_tensors='pt')
    if device == 'cuda':
        inputs = {k: v.to('cuda') for k, v in inputs.items()}

    input_tokens = inputs['input_ids'].shape[1]

    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            do_sample=(temperature > 0.0),
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    if device == 'cuda':
        torch.cuda.synchronize()
    latency = time.perf_counter() - t0

    gen_tokens = outputs[0][input_tokens:]
    out_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    tok_per_sec = len(gen_tokens) / max(latency, 0.001)

    return {
        'response': out_text,
        'latency_seconds': round(latency, 3),
        'input_tokens': input_tokens,
        'output_tokens': len(gen_tokens),
        'tokens_per_second': round(tok_per_sec, 2)
    }


In [ ]:
# Step 5: Execute Evaluation Across Held-Out Test Set (38 cases)
import json
import os
import urllib.request
from pathlib import Path

# Multi-tier candidate resolution for Google Colab and local environments
candidate_paths = [
    Path('data/processed/test.jsonl'),
    Path('data/test.jsonl'),
    Path('/content/python-learning-assistant-slm/data/processed/test.jsonl'),
    Path('/content/python-learning-assistant-slm/data/test.jsonl'),
    Path('/content/slm-python-learning-assistant/data/processed/test.jsonl'),
    Path('/content/slm-python-learning-assistant/data/test.jsonl'),
]

test_file = None
for p in candidate_paths:
    if p.exists() and p.is_file():
        test_file = p
        break

# Auto-recovery for Google Colab if executing in /content without cloned repo
if test_file is None:
    print('Held-out test file not found in local paths. Attempting Colab workspace recovery...')
    # 1. Attempt git clone if running in Colab
    colab_repo_dir = Path('/content/python-learning-assistant-slm')
    if not colab_repo_dir.exists():
        try:
            import subprocess
            print('Cloning repository into /content/python-learning-assistant-slm...')
            subprocess.run(
                ['git', 'clone', 'https://github.com/sasikalascgm-droid/python-learning-assistant-slm.git', str(colab_repo_dir)],
                check=True
            )
            if (colab_repo_dir / 'data/processed/test.jsonl').exists():
                test_file = colab_repo_dir / 'data/processed/test.jsonl'
                os.chdir(colab_repo_dir)
                print(f'Switched working directory to: {colab_repo_dir}')
        except Exception as e:
            print(f'Git clone fallback encountered: {e}')
    
    # 2. Direct GitHub download fallback if clone was not possible
    if test_file is None or not test_file.exists():
        fallback_dir = Path('data/processed')
        fallback_dir.mkdir(parents=True, exist_ok=True)
        test_file = fallback_dir / 'test.jsonl'
        raw_url = 'https://raw.githubusercontent.com/sasikalascgm-droid/python-learning-assistant-slm/master/data/processed/test.jsonl'
        print(f'Downloading official held-out test.jsonl from GitHub: {raw_url}...')
        urllib.request.urlretrieve(raw_url, test_file)
        print(f'Successfully downloaded test dataset to: {test_file}')

if not test_file or not test_file.exists():
    raise FileNotFoundError(
        f'FATAL: Held-out test dataset could not be located. Searched candidates: {[str(p) for p in candidate_paths]}'
    )

print(f'Loading held-out test cases from: {test_file.resolve()}')

test_cases = []
with open(test_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            test_cases.append(json.loads(line))

assert len(test_cases) == 38, f'Expected exactly 38 held-out test cases, but found {len(test_cases)}!'
print(f'Successfully loaded {len(test_cases)} held-out test cases (test set strictly verified).')

results = []
for idx, tc in enumerate(test_cases, 1):
    inst = tc['instruction']
    inp = tc.get('input', '')
    gt = tc.get('output', '')
    
    print(f'[{idx}/{len(test_cases)}] Evaluating: {inst[:45]}...')
    out = generate_slm_response(inst, inp)
    
    results.append({
        'test_case_id': idx,
        'instruction': inst,
        'input': inp,
        'ground_truth': gt,
        'model_response': out['response'],
        'latency_seconds': out['latency_seconds'],
        'input_tokens': out['input_tokens'],
        'output_tokens': out['output_tokens'],
        'tokens_per_second': out['tokens_per_second'],
        'status': 'success'
    })

print('Evaluation complete!')


In [ ]:
# Step 6: Summary Metrics and Export to experiments/baseline_results.json
import time
from pathlib import Path

latencies = [r['latency_seconds'] for r in results]
tokens = [r['output_tokens'] for r in results]
tps = [r['tokens_per_second'] for r in results]
tot_lat = sum(latencies)
tot_tok = sum(tokens)

summary = {
    'evaluation_name': 'Baseline SLM Evaluation (Unadapted Qwen2.5-1.5B-Instruct)',
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'model_id': MODEL_ID,
    'model_revision': PINNED_REVISION,
    'device': device,
    'precision': str(dtype),
    'total_test_cases': len(results),
    'successful_cases': len(results),
    'failed_cases': 0,
    'total_tokens_generated': tot_tok,
    'total_generation_time_seconds': round(tot_lat, 2),
    'true_aggregate_throughput_tok_per_sec': round(tot_tok / max(tot_lat, 0.001), 2),
    'average_latency_seconds': round(tot_lat / len(latencies), 3),
    'median_latency_seconds': round(sorted(latencies)[len(latencies)//2], 3),
    'average_tokens_generated': round(tot_tok / len(tokens), 1),
    'average_tokens_per_second': round(sum(tps) / len(tps), 2),
    'test_results': results
}

out_dir = Path('experiments')
out_dir.mkdir(parents=True, exist_ok=True)
results_file = out_dir / 'baseline_results.json'
with open(results_file, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f'Saved {len(results)} evaluated test cases to {results_file}')
print(f'Average Latency: {summary["average_latency_seconds"]}s | True Aggregate Throughput: {summary["true_aggregate_throughput_tok_per_sec"]} tok/s')

# Optional browser download if executing in Google Colab
try:
    from google.colab import files
    print('\nPrompting browser download of experiments/baseline_results.json...')
    files.download(str(results_file))
except (ImportError, Exception):
    print(f'\nResults file is ready at: {results_file.resolve()}')
